# Problem 1: Collaborative Filtering for Recommending Songs to Music Listeners

* Recommendation  engines  are  vital  to  many  of  today’s  most  successful  firms.   From  Amazon  to Netflix to Spotify, companies have realized the value of automatically showing customers products they are more interested in purchasing based on the collaborative evaluations of products by customers,  and  based  also  on  customer-specific  and  product-specific  data  as  well.   

* In  this  problem, we provide you with data consisting of derived user ratings of songs based on each user’s listening behavior, obtained from the Million Song Dataset Project.  (Note that users do not actually rate songs – rather the “rating” has been determined from their listening behaviors – but we will stil luse phrases like “user 'i' has rated song 'j' ”, etc.)  

* Each user only listens to a few songs, and therefore for each user we only have rating data concerning these few songs.  Nevertheless, we would like to infer each user’s ratings of all songs based on the entirety of the data.  

**The summary goal of this problem will be to recommend specific songs to specific users.**

* The  data  for  this  problem  has  already  been  split  into  training,  validation,  and  test  sets  and  is provided in four .csv files: **MusicRatingsTrain.csv(85%)**, **MusicRatingsValidationA.csv(5%)**, **MusicRatingsValidationB.csv(5%)**, and **MusicRatingsTest.csv(5%)**.  

**Validation set A will be used for  hyperparameter  tuning  of  individual  models,  and  validation  set  B  will  be  used  for  blending.**

* Each observation corresponds to a user who has rated a particular song.  Table 1 below describes the columns of this data.

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import statistics 

### Importing the data

In [2]:
music_ratings_train = pd.read_csv('MusicRatingsTrain.csv')
music_ratings_test = pd.read_csv('MusicRatingsTest.csv')
music_ratings_ValA = pd.read_csv('MusicRatingsValidationA.csv')
music_ratings_ValB = pd.read_csv('MusicRatingsValidationB.csv')

In [3]:
music_ratings_train

,userID,songID,rating,songName,year,artist,genre
0,2071,581,2.484754,Show You How,2004,The Killers,Rock
1,1208,55,1.000000,Luvstruck,1999,Southside Spinners,Electronic
2,1484,510,1.000000,In One Ear,2008,Cage The Elephant,Rock
3,453,299,1.000000,Gardenhead / Leave Me Alone,1992,Neutral Milk Hotel,Rock
4,2238,113,1.000000,Hustler,2006,Simian Mobile Disco,Electronic
...,...,...,...,...,...,...,...
245992,1513,720,1.000000,Trani,2003,Kings Of Leon,Rock
245993,443,544,1.000000,A Rush Of Blood To The Head,2002,Coldplay,Rock
245994,2172,706,1.784426,Without Me,2002,Eminem,Rap
245995,1714,699,1.494918,Cherbourg,2007,Beirut,Folk


In [4]:
music_ratings_train.shape

(245997, 7)

In [5]:
music_ratings_test

,userID,songID,rating,songName,year,artist,genre
0,853,54,1.989836,You're The One,1990,Dwight Yoakam,Country
1,608,34,1.000000,Nah!,2002,Shania Twain,Country
2,9,54,3.969507,You're The One,1990,Dwight Yoakam,Country
3,1862,54,2.279344,You're The One,1990,Dwight Yoakam,Country
4,329,80,1.494918,Tim McGraw,2006,Taylor Swift,Country
...,...,...,...,...,...,...,...
14466,443,615,1.494918,Daylight,2002,Coldplay,Rock
14467,1061,657,1.000000,Kissy Kissy,2003,The Kills,Rock
14468,610,653,1.000000,The Laws Have Changed,2003,The New Pornographers,Rock
14469,690,657,1.000000,Kissy Kissy,2003,The Kills,Rock


In [6]:
music_ratings_test.shape

(14471, 7)

In [7]:
music_ratings_ValA

,userID,songID,rating,songName,year,artist,genre
0,1420,787,1.000000,Ragoo,2007,Kings Of Leon,Rock
1,2110,503,1.494918,In The End (Album Version),2000,Linkin Park,Rock
2,513,190,1.989836,Not Fair (Clean Radio Edit),2009,Lily Allen,Pop
3,1323,688,1.494918,A Message,2005,Coldplay,Rock
4,431,467,1.000000,I Will Follow You into the Dark (Album Version),2005,Death Cab for Cutie,Rock
...,...,...,...,...,...,...,...
14465,1118,721,1.000000,Milk,2004,Kings Of Leon,Country
14466,442,59,1.000000,They Might Follow You,2007,Tiny Vipers,Pop
14467,1975,217,1.000000,One More Sad Song,2001,The All-American Rejects,Rock
14468,1712,472,1.494918,Anything New,2007,Digitalism,Electronic


In [8]:
music_ratings_ValA.shape

(14470, 7)

In [9]:
music_ratings_ValB

,userID,songID,rating,songName,year,artist,genre
0,2102,13,1.494918,Such Great Heights,2003,The Postal Service,Electronic
1,1350,320,1.000000,The Shrew,2009,Beirut,Folk
2,947,7,1.494918,Halo,2008,Beyonce,Pop
3,175,558,1.989836,The Middle,2001,Jimmy Eat World,Rock
4,1984,445,1.494918,Daddy,2003,Beyonce,RnB
...,...,...,...,...,...,...,...
14466,1268,712,1.494918,Street Justice,2006,MSTRKRFT,Rock
14467,1154,575,4.480967,Hide & Seek,2009,Imogen Heap,Pop
14468,1640,367,1.000000,Holy Roller Novocaine,2003,Kings Of Leon,Rock
14469,2358,721,1.000000,Milk,2004,Kings Of Leon,Country


In [10]:
music_ratings_ValB.shape

(14471, 7)

#### Sorting the datasets by userId and songID

In [11]:
music_ratings_train = music_ratings_train.sort_values(by=['userID', 'songID'])

In [12]:
music_ratings_train.head()

,userID,songID,rating,songName,year,artist,genre
116572,1,10,1.784426,Harder Better Faster Stronger,2007,Daft Punk,Electronic
63286,1,22,1.000000,Up Up & Away,2009,Kid Cudi,Rap
219232,1,26,3.451920,Undo,2001,Bjork,Rock
243057,1,34,1.000000,Nah!,2002,Shania Twain,Country
103269,1,35,1.784426,Imma Be,2009,Black Eyed Peas,Rap


In [13]:
music_ratings_test = music_ratings_test.sort_values(by=(['userID', 'songID']))
music_ratings_test.head()

,userID,songID,rating,songName,year,artist,genre
3187,1,7,1.989836,Halo,2008,Beyonce,Pop
2211,1,753,1.784426,Bitter Sweet Symphony,1997,The Verve,Pop
2837,2,72,1.784426,Teeth,2009,Lady GaGa,Pop
11793,2,481,1.000000,Waterfall,1978,Triumvirat,Rock
12493,2,507,1.784426,Yellow,2000,Coldplay,Rock


In [14]:
music_ratings_ValA = music_ratings_ValA.sort_values(by=['userID', 'songID'])
music_ratings_ValA.head()

,userID,songID,rating,songName,year,artist,genre
364,1,25,1.000000,Seven Nation Army,2003,The White Stripes,Rock
12933,1,77,1.000000,Enter Sandman,1991,Metallica,Rock
11337,1,600,3.404302,Revelry,2008,Kings Of Leon,Rock
3036,1,635,2.484754,Make Love To Your Mind,1975,Bill Withers,RnB
1311,2,22,2.831414,Up Up & Away,2009,Kid Cudi,Rap


In [15]:
music_ratings_ValB = music_ratings_ValB.sort_values(by=['userID', 'songID'])
music_ratings_ValB.head()

,userID,songID,rating,songName,year,artist,genre
2523,1,93,1.989836,Clocks,2002,Coldplay,Rock
3773,1,462,1.000000,Rabbit Heart (Raise It Up),2009,Florence + The Machine,Rock
1777,1,663,1.784426,Tighten Up,2010,The Black Keys,Electronic
7897,2,24,1.000000,Swallowed In The Sea,2005,Coldplay,Electronic
1956,2,47,1.000000,I Stand Corrected (Album),2007,Vampire Weekend,Rock


## 1a) 
**How  many  songs  are  in  the  training  dataset?   How  many  users?   Based  on  the training set, provide summary statistics (mean, median, standard deviation, min, and max) for the rating variable.**

In [17]:
songs_train_len = len(music_ratings_train['songID'].unique())
print(f'There are {songs_train_len} unique songs in the training dataset.')

There are 807 unique songs in the training dataset.


In [18]:
users_train_len = len(music_ratings_train['userID'].unique())
print(f'There are {users_train_len} unique users in the training dataset.')

There are 2421 unique users in the training dataset.


In [21]:
# Summary Statistics Table for Rating Variable
music_ratings_train.describe()[['rating']]

,rating
count,245997.000000
mean,1.321962
std,0.459894
min,1.000000
25%,1.000000
50%,1.000000
75%,1.494918
max,4.768656


In [23]:
print(f'Mean of the rating variable is {music_ratings_train.describe()['rating']['mean']}')

Mean of the rating variable is 1.3219620891310984


In [26]:
print(f'Median of the rating variable is {music_ratings_train.describe()['rating']['50%']}')

Median of the rating variable is 1.0


In [27]:
print(f'Standard Devation of the rating variable is {music_ratings_train.describe()['rating']['std']}')

Standard Devation of the rating variable is 0.45989367021737887


In [28]:
print(f'Min of the rating variable is {music_ratings_train.describe()['rating']['min']}')

Min of the rating variable is 1.0


In [29]:
print(f'Max of the rating variable is {music_ratings_train.describe()['rating']['max']}')

Max of the rating variable is 4.768655987487031


## 1b)
Let **X** denote the “complete” ratings matrix, i.e., **Xi,j** denotes either the observed rating if user 'i' actually rated song 'j' or the “hypothetical” such rating if user 'i' has not yet rated song 'j'.  We are interested in predicting the values of **Xi,j** that are not observed.  

Let us first consider the following model: **Xi,j = αi + βj + epsilon i,j,**

where **αi** is a coefficient that depends only on the particular row **i**(i.e., user), **βj** is a coefficient that depends only on the particular column **j**(i.e., song), and **epsilon i,j** is a noise term.

### i)
**For this dataset, how many total parameters are included in model (1)?  How many observations do we have to train the model with?**

**Use BiScaler to fit model (1) on the training data.** 

In [30]:
print(f'There are a total of {songs_train_len + users_train_len} parameters in model(1).')

There are a total of 3228 parameters in model(1).


In [31]:
print(f'We have to train the model on {len(music_ratings_train)} observations.')

We have to train the model on 245997 observations.


In [34]:
# %pip install fancyimpute

In [35]:
from fancyimpute import BiScaler

### Training Set Pre-Processing

In [36]:
# Original trianing set.
music_ratings_train.sort_index()

,userID,songID,rating,songName,year,artist,genre
0,2071,581,2.484754,Show You How,2004,The Killers,Rock
1,1208,55,1.000000,Luvstruck,1999,Southside Spinners,Electronic
2,1484,510,1.000000,In One Ear,2008,Cage The Elephant,Rock
3,453,299,1.000000,Gardenhead / Leave Me Alone,1992,Neutral Milk Hotel,Rock
4,2238,113,1.000000,Hustler,2006,Simian Mobile Disco,Electronic
...,...,...,...,...,...,...,...
245992,1513,720,1.000000,Trani,2003,Kings Of Leon,Rock
245993,443,544,1.000000,A Rush Of Blood To The Head,2002,Coldplay,Rock
245994,2172,706,1.784426,Without Me,2002,Eminem,Rap
245995,1714,699,1.494918,Cherbourg,2007,Beirut,Folk


In [38]:
# Use pivot function so that each column is a unique song and each row is a unique user
music_ratings_train_df = music_ratings_train.pivot(index="userID", columns="songID", values="rating")
music_ratings_train_df

songID,1,2,3,4,5,6,7,8,9,10,...,798,799,800,801,802,803,804,805,806,807
userID,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.784426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1.989836,NaN,NaN,NaN,NaN,1.784426,NaN,NaN,1.784426,...,NaN,1.784426,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,1.494918,NaN,NaN,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
5,NaN,1.000000,NaN,NaN,NaN,NaN,2.389410,NaN,NaN,2.279344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2417,NaN,NaN,NaN,1.0,NaN,NaN,1.000000,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2418,NaN,NaN,NaN,NaN,NaN,NaN,1.494918,NaN,NaN,1.989836,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2419,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,1.000000,...,NaN,NaN,NaN,1.494918,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# Convert training set to matrix form
music_ratings_train_mat = music_ratings_train_df.to_numpy()
music_ratings_train_mat

array([[       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan, 1.98983574,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan, 1.49491787, 1.49491787, ...,        nan,        nan,
               nan]], shape=(2421, 807))

In [40]:
# Making sure the number of users and songs are the same as in the training set.
print(len(np.unique(music_ratings_train.userID)))
print(music_ratings_train_mat.shape)

2421
(2421, 807)


### The Simple Additive Model through BiScaler on the Training Set

In [41]:
# The Simple Additive Model through BiScaler
music_ratings_biscaler = BiScaler(scale_rows=False, scale_columns=False, verbose=True)
music_ratings_train_mat_centered = music_ratings_biscaler.fit_transform(music_ratings_train_mat)

[BiScaler] Initial log residual value = 8.635876
[BiScaler] Iter 1: log residual = -1.278081, log improvement ratio=9.913958
[BiScaler] Iter 2: log residual = -2.464318, log improvement ratio=1.186237
[BiScaler] Iter 3: log residual = -3.599288, log improvement ratio=1.134970
[BiScaler] Iter 4: log residual = -4.728969, log improvement ratio=1.129681
[BiScaler] Iter 5: log residual = -5.851105, log improvement ratio=1.122136
[BiScaler] Iter 6: log residual = -6.961307, log improvement ratio=1.110202
[BiScaler] Iter 7: log residual = -8.052791, log improvement ratio=1.091484
[BiScaler] Iter 8: log residual = -9.115833, log improvement ratio=1.063043
[BiScaler] Iter 9: log residual = -10.137984, log improvement ratio=1.022150
[BiScaler] Iter 10: log residual = -11.105953, log improvement ratio=0.967969
[BiScaler] Iter 11: log residual = -12.009465, log improvement ratio=0.903512
[BiScaler] Iter 12: log residual = -12.845445, log improvement ratio=0.835981
[BiScaler] Iter 13: log residual

In [42]:
# Access the alpha (row_mean) and beta (column_mean) values
print(music_ratings_biscaler.row_means)
print(music_ratings_biscaler.column_means)

[ 0.23597732  0.37836219 -0.12202861 ... -0.00099498 -0.09292935
 -0.07297946]
[1.1085277  1.28189302 1.45665654 1.15972853 1.28211814 1.12983062
 1.47101127 1.28577961 1.35033994 1.43749681 1.16378442 1.21511576
 1.4097627  1.23517437 1.25240525 1.18324648 1.57097994 1.24492046
 1.50054534 1.21816032 1.19678    1.45653583 1.23245392 1.1058456
 1.41882857 2.1370632  1.40920159 1.12776456 1.32721814 1.21159835
 1.11964289 1.20034543 1.17693884 1.58421436 1.45240405 1.27537058
 1.23174929 1.15390764 1.29276939 1.11644992 1.37531285 1.14228572
 1.21817148 1.26869206 1.20196407 1.18522847 1.2132076  1.20720276
 1.12599004 1.17277644 1.26637501 1.19294522 1.16589595 2.15784144
 1.7676486  1.19068097 1.12156866 1.13149055 1.85976324 1.17344651
 1.41308493 1.10785718 1.09833489 1.35995412 1.31486195 1.2667538
 1.37337888 1.11536594 1.24306441 1.45978565 1.16395292 1.21780274
 1.26713043 1.20356712 1.20973988 1.4430925  1.23889642 1.3110657
 1.39755168 1.35581768 1.15864186 1.19324072 1.134301

#### Fitting the Simple Additive Model on the training data

In [43]:
import copy
# Copy of CENTERED matrix
music_ratings_train_mat_centered_0 = copy.copy(music_ratings_train_mat_centered)
# Replace missing values with 0, because in simple additive model, missing values
music_ratings_train_mat_centered_0[np.isnan(music_ratings_train_mat_centered_0)]=0
music_ratings_train_mat_centered_0

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.32958053, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.28600431, 0.11124078, ..., 0.        , 0.        ,
        0.        ]], shape=(2421, 807))

In [44]:
# Use inverse_transform to Transform centered table to the original table
music_ratings_BiScaler_filled = music_ratings_biscaler.inverse_transform(music_ratings_train_mat_centered_0)
music_ratings_BiScaler_filled

array([[1.34450503, 1.51787035, 1.69263387, ..., 1.55709616, 1.41842513,
        1.40129053],
       [1.4868899 , 1.98983574, 1.83501874, ..., 1.69948103, 1.56081   ,
        1.5436754 ],
       [0.98649909, 1.15986441, 1.33462793, ..., 1.19909023, 1.0604192 ,
        1.0432846 ],
       ...,
       [1.10753272, 1.28089804, 1.45566156, ..., 1.32012386, 1.18145283,
        1.16431823],
       [1.01559836, 1.18896367, 1.3637272 , ..., 1.22818949, 1.08951846,
        1.07238386],
       [1.03554825, 1.49491787, 1.49491787, ..., 1.24813938, 1.10946835,
        1.09233375]], shape=(2421, 807))

### ii)
**Using the output of the BiScaler, what are the three most popular songs after  removing  for  the  bias  due  to  users’  affinity  for  rating  songs  highly  (or  lowly)?**

**Provide the song ID numbers as well as the song and artist names for these three songs. Explain, in your own words, how your answer relates to model (1).**

In [45]:
music_ratings_train_mat_centered

array([[       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan, 0.32958053,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan, 0.28600431, 0.11124078, ...,        nan,        nan,
               nan]], shape=(2421, 807))

In [46]:
music_ratings_BiScaler_filled

array([[1.34450503, 1.51787035, 1.69263387, ..., 1.55709616, 1.41842513,
        1.40129053],
       [1.4868899 , 1.98983574, 1.83501874, ..., 1.69948103, 1.56081   ,
        1.5436754 ],
       [0.98649909, 1.15986441, 1.33462793, ..., 1.19909023, 1.0604192 ,
        1.0432846 ],
       ...,
       [1.10753272, 1.28089804, 1.45566156, ..., 1.32012386, 1.18145283,
        1.16431823],
       [1.01559836, 1.18896367, 1.3637272 , ..., 1.22818949, 1.08951846,
        1.07238386],
       [1.03554825, 1.49491787, 1.49491787, ..., 1.24813938, 1.10946835,
        1.09233375]], shape=(2421, 807))

In [47]:
len(music_ratings_BiScaler_filled[0])

807

In [48]:
len(music_ratings_BiScaler_filled)

2421

In [49]:
# Transpose the matrix so that we can get each song into its own array.
music_ratings_BiScaler_filled.T

array([[1.34450503, 1.4868899 , 0.98649909, ..., 1.10753272, 1.01559836,
        1.03554825],
       [1.51787035, 1.98983574, 1.15986441, ..., 1.28089804, 1.18896367,
        1.49491787],
       [1.69263387, 1.83501874, 1.33462793, ..., 1.45566156, 1.3637272 ,
        1.49491787],
       ...,
       [1.55709616, 1.69948103, 1.19909023, ..., 1.32012386, 1.22818949,
        1.24813938],
       [1.41842513, 1.56081   , 1.0604192 , ..., 1.18145283, 1.08951846,
        1.10946835],
       [1.40129053, 1.5436754 , 1.0432846 , ..., 1.16431823, 1.07238386,
        1.09233375]], shape=(807, 2421))

#### Summing up the rating for each song

In [50]:
# Summing up the rating for each song and storing the total in a list
song_total_ratings = []
for i in range(len(music_ratings_BiScaler_filled.T)):
    song_total_ratings.append(sum(music_ratings_BiScaler_filled.T[i]))

In [51]:
# Every rating for each song available summed up.
song_total_ratings

[np.float64(2650.306372711403),
 np.float64(3070.0238087837743),
 np.float64(3493.1262959372984),
 np.float64(2774.263563987291),
 np.float64(3070.568820597366),
 np.float64(2701.8807254648764),
 np.float64(3527.8790868388273),
 np.float64(3079.4332260406804),
 np.float64(3235.7337990530755),
 np.float64(3446.740576932402),
 np.float64(2784.0828785063345),
 np.float64(2908.356044714126),
 np.float64(3379.5963028820775),
 np.float64(2956.9179584669178),
 np.float64(2998.6339049813137),
 np.float64(2831.200538593642),
 np.float64(3769.903243040136),
 np.float64(2980.513245058633),
 np.float64(3599.3810677951806),
 np.float64(2915.726931170478),
 np.float64(2863.965193045014),
 np.float64(3492.834050771581),
 np.float64(2950.3317461219435),
 np.float64(2643.813001663041),
 np.float64(3401.544758217399),
 np.float64(5140.390816224416),
 np.float64(3378.237841785364),
 np.float64(2696.8787893813324),
 np.float64(3179.755929265449),
 np.float64(2899.8404114300247),
 np.float64(2677.216241511

In [52]:
# The top 3 highest ratings
top_three_ratings = sorted(song_total_ratings, reverse=True)[:3]
top_three_ratings

[np.float64(5190.694920507644),
 np.float64(5140.390816224416),
 np.float64(4901.616334352098)]